In [1]:
# %% [markdown]
# # libraries and env (Unchanged)

# %%
import pandas as pd
import os
from pathlib import Path
import pickle

# %%
from dotenv import load_dotenv

def load_environment():
    """Load environment variables and return them as a dictionary."""
    load_dotenv(Path("../utils/.env"))  
    env_vars = {
        "EXTRACTEDLANDCOVER_FOLDER": os.getenv("EXTRACTEDLANDCOVER_FOLDER"),
        "CLEANEDLANDCOVER_FOLDER": os.getenv("CLEANEDLANDCOVER_FOLDER")
    }
    return env_vars

folders = load_environment()
extracted_landcover_folder = folders["EXTRACTEDLANDCOVER_FOLDER"]
cleaned_landcover_folder = folders["CLEANEDLANDCOVER_FOLDER"]

# Ensure the output directory exists
Path(cleaned_landcover_folder).mkdir(parents=True, exist_ok=True)

# Define file paths
PARQUET_IN_PATH = f"{extracted_landcover_folder}/landcover_0p1_grid.parquet"
PARQUET_OUT_PATH = f"{cleaned_landcover_folder}/landcover_0p1_grid_cleaned.parquet"
MAPPING_PATH = f"{extracted_landcover_folder}/lcc_highlevel_mapping.pkl"


# %% [markdown]
# # Read the Tabular Grid File

# %%
# Reading the Land Cover dataset Parquet file
try:
    # Use the grid file containing 'cell_id'
    landcover_df = pd.read_parquet(PARQUET_IN_PATH)
    print(f"✅ Successfully read {len(landcover_df)} rows from {os.path.basename(PARQUET_IN_PATH)}")
except FileNotFoundError:
    print(f"❌ Error: File not found at {PARQUET_IN_PATH}. Please ensure the previous step ran correctly.")
    raise

# Display initial columns to verify attributes
print("Initial Columns:", landcover_df.columns.tolist())
print(landcover_df.head())

# %% [markdown]
# # Cleaning and Feature Transformation

# %%
# --- 1. Drop unwanted columns (gridcode and id) ---
columns_to_drop = ["gridcode", "id"] # Assuming 'id' is the column that needs to be dropped (check your data if it's named differently)

# Check if columns exist before dropping to prevent errors
for col in columns_to_drop:
    if col in landcover_df.columns:
        landcover_df = landcover_df.drop(columns=[col])
        print(f"Dropped column: {col}")
    else:
        print(f"Column {col} not found, skipping drop.")


# %%
# --- 2. Transform lcccode to High-Level Categories ---

# Read pkl file (lcc_highlevel_mapping)
try:
    with open(MAPPING_PATH, "rb") as f:
        lcc_highlevel_mapping = pickle.load(f)
    print("✅ Loaded LCC high-level mapping.")
except FileNotFoundError:
    print(f"❌ Error: Mapping file not found at {MAPPING_PATH}. Cannot transform lcccode.")
    raise

# Add lcc_highlevel column to landcover_df
if "lcccode" in landcover_df.columns:
    landcover_df["lcc_highlevel"] = landcover_df["lcccode"].map(lcc_highlevel_mapping)
    print("Added 'lcc_highlevel' column.")
else:
    print("⚠️ Column 'lcccode' not found. Cannot perform mapping.")

# Handle Nulls (Optional: check if any codes were missed in the mapping)
null_count = landcover_df["lcc_highlevel"].isnull().sum()
if null_count > 0:
    print(f"\nFound {null_count} cells with unmapped 'lcccode' (NaN in lcc_highlevel).")
    unmapped_codes = landcover_df[landcover_df["lcc_highlevel"].isnull()]["lcccode"].unique()
    print("Unmapped lcccodes:", unmapped_codes)
    # You might consider filling these NaNs with "Unclassified" or similar based on your domain knowledge.
    landcover_df["lcc_highlevel"] = landcover_df["lcc_highlevel"].fillna("Unclassified")
else:
    print("No null values found in 'lcc_highlevel'.")


# %%
# --- 3. Drop the original lcccode column ---
if "lcccode" in landcover_df.columns:
    landcover_df = landcover_df.drop(columns=["lcccode"])
    print("Dropped original 'lcccode' column.")

print("\nFinal Cleaned DataFrame Head:")
print(landcover_df.head())
print("Final Columns:", landcover_df.columns.tolist())


# %% [markdown]
# # Save the Cleaned File

# %%
# Saving the cleaned landcover DataFrame to a new parquet file
try:
    landcover_df.to_parquet(PARQUET_OUT_PATH)
    print(f"✅ Saved cleaned land cover grid to:\n{PARQUET_OUT_PATH}")
except Exception as e:
    # Adding a robust fallback just in case of further Parquet/Arrow errors
    CSV_OUT_PATH = PARQUET_OUT_PATH.replace(".parquet", ".csv")
    landcover_df.to_csv(CSV_OUT_PATH, index=False)
    print(f"⚠️ Parquet failed (Error: {e}). Saved as CSV instead to:\n{CSV_OUT_PATH}")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\

AttributeError: _ARRAY_API not found

✅ Successfully read 23322 rows from landcover_0p1_grid.parquet
Initial Columns: ['cell_id', 'id', 'gridcode', 'area', 'lcccode']
       cell_id        id  gridcode          area lcccode
index                                                   
0            0  111056.0     201.0  3.358561e+11    6001
1            1  111056.0     201.0  3.358561e+11    6001
2            2  111056.0     201.0  3.358561e+11    6001
3            3  111056.0     201.0  3.358561e+11    6001
4            4  111056.0     201.0  3.358561e+11    6001
Dropped column: gridcode
Dropped column: id
✅ Loaded LCC high-level mapping.
Added 'lcc_highlevel' column.

Found 133 cells with unmapped 'lcccode' (NaN in lcc_highlevel).
Unmapped lcccodes: [None]
Dropped original 'lcccode' column.

Final Cleaned DataFrame Head:
       cell_id          area lcc_highlevel
index                                     
0            0  3.358561e+11    Bare lands
1            1  3.358561e+11    Bare lands
2            2  3.358561e+11    Bare


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\

AttributeError: _ARRAY_API not found

✅ Saved cleaned land cover grid to:
../../CleanedDatasets/LandCoverDataset/landcover_0p1_grid_cleaned.parquet
